# MOM6 surface fields: storm GIF

Read the archived MOM6 `mom6.h.sfc` history files and animate surface current speed (`speed`), sea surface temperature (`tos`), and surface salinity (`sos`). Colors stay fixed across frames. October 7–12, 2024 is shown at every available output time and held longer in the GIF.

Run the cells from top to bottom. Change only the settings below for another case. This notebook reads archived data and writes one GIF; it does not change the case.

**Environment (CLI, only if packages are missing):** `python -m pip install xarray netCDF4 cftime matplotlib pillow numpy`


In [31]:
from pathlib import Path
from datetime import date, datetime
from io import BytesIO

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from PIL import Image

# User settings
ARCHIVE_DIR = Path("/glade/derecho/scratch/hmnguyen/archive/florida_storm_cam6_test")
CASE_NAME = "florida_storm_cam6_test"             # None: infer from the archive; or set the exact case name
OUTPUT_DIR = Path.cwd() / "animations"
OUTPUT_NAME = None           # None: <case>_surface_storm.gif
DATE_START = date(2024, 10, 1)            # e.g. date(2024, 9, 20); None uses the first archived record
DATE_END = date(2024, 10, 31)              # e.g. date(2024, 10, 20); inclusive

# October 7–12 is inclusive. Available storm records are all retained.
STORM_START = date(2024, 10, 7)
STORM_END = date(2024, 10, 12)
OUTSIDE_TARGET_HOURS = 1    # sample one frame about every 1 h outside the storm
OUTSIDE_FRAME_MS = 120      # GIF delay per frame outside the storm
STORM_FRAME_MS = 250        # GIF delay per frame during the storm
FIGURE_DPI = 100

assert ARCHIVE_DIR.is_dir(), f"Archive directory does not exist: {ARCHIVE_DIR}"
assert STORM_START <= STORM_END
assert OUTSIDE_TARGET_HOURS > 0 and OUTSIDE_FRAME_MS > 0 and STORM_FRAME_MS > 0


## Find the archived surface history

The search is recursive because CESM archives commonly place ocean history under `ocn/hist`. Files from different months are combined by their decoded `time` coordinate, rather than by filename order alone.


In [32]:
def surface_case_name(path):
    marker = ".mom6.h.sfc"
    return path.name.split(marker, 1)[0] if marker in path.name else None


sfc_files = sorted(
    p for p in ARCHIVE_DIR.rglob("*.nc")
    if ".mom6.h.sfc" in p.name
    and p.name.endswith(".nc")
    and (CASE_NAME is None or surface_case_name(p) == CASE_NAME)
)
if not sfc_files:
    raise FileNotFoundError(f"No MOM6 sfc NetCDF files found below {ARCHIVE_DIR}")

case_names = {surface_case_name(p) for p in sfc_files}
if len(case_names) != 1:
    raise ValueError(f"Several cases found: {sorted(case_names)}. Set CASE_NAME in the first cell.")
case_name = case_names.pop()

for path in sfc_files:
    print(path)
print(f"Case: {case_name} | files: {len(sfc_files)}")


/glade/derecho/scratch/hmnguyen/archive/florida_storm_cam6_test/ocn/hist/florida_storm_cam6_test.mom6.h.sfc.nc
Case: florida_storm_cam6_test | files: 1


In [33]:
def as_datetime(value):
    # Handles decoded numpy.datetime64, pandas timestamps and cftime values.
    return datetime.fromisoformat(str(value).replace("T", " ")[:19])


records = []
variable_names = ("speed", "tos", "sos")
for path in sfc_files:
    with xr.open_dataset(path, decode_times=True, mask_and_scale=True) as ds:
        missing = set(variable_names) - set(ds.data_vars)
        if missing:
            raise KeyError(f"{path}: missing {sorted(missing)}")
        if "time" not in ds.coords:
            raise KeyError(f"{path}: no decoded time coordinate")
        for index, value in enumerate(ds.time.values):
            when = as_datetime(value)
            if DATE_START is not None and when.date() < DATE_START:
                continue
            if DATE_END is not None and when.date() > DATE_END:
                continue
            records.append((when, path, index))

records.sort(key=lambda record: record[0])
unique_records = {}
for record in records:
    unique_records.setdefault(record[0], record)  # shared monthly boundary, if present
duplicates = len(records) - len(unique_records)
records = list(unique_records.values())
if not records:
    raise ValueError("No sfc records are inside DATE_START–DATE_END")

selected = []
last_outside = None
for record in records:
    when = record[0]
    in_storm = STORM_START <= when.date() <= STORM_END
    if in_storm or last_outside is None or (when - last_outside).total_seconds() >= 3600 * OUTSIDE_TARGET_HOURS:
        selected.append(record)
        if not in_storm:
            last_outside = when
if selected[-1][0] != records[-1][0]:
    selected.append(records[-1])

print(f"Archive time: {records[0][0]} to {records[-1][0]}")
print(f"Selected {len(selected)} / {len(records)} records; duplicate times removed: {duplicates}")
print(f"Storm frames: {sum(STORM_START <= t.date() <= STORM_END for t, _, _ in selected)}")


Archive time: 2024-10-01 00:00:00 to 2024-10-30 00:00:00
Selected 513 / 697 records; duplicate times removed: 0
Storm frames: 144


## Plot coordinates and missing values

Use the case's `ocean_geometry.nc` geographic cell centers when available. Otherwise use the surface file's `xh`/`yh` axes. MOM6 missing values can be very large even when a file's `_FillValue` differs; these and dry cells are masked before estimating colors or drawing frames.


In [28]:
with xr.open_dataset(sfc_files[0]) as first:
    spatial_dims = set(first["tos"].dims) - {"time"}
    y_dim = next((d for d in ("yh", "lath", "lat", "latitude", "y") if d in spatial_dims), None)
    x_dim = next((d for d in ("xh", "lonh", "lon", "longitude", "x") if d in spatial_dims), None)
    if y_dim is None or x_dim is None or spatial_dims != {y_dim, x_dim}:
        raise ValueError(f"Cannot identify the two sfc spatial axes: {first['tos'].dims}")
    example = first["tos"].isel(time=0)
    ny, nx = example.transpose(y_dim, x_dim).shape
    x_axis = np.asarray(first[x_dim].values) if x_dim in first.coords else np.arange(nx)
    y_axis = np.asarray(first[y_dim].values) if y_dim in first.coords else np.arange(ny)
    units = {name: first[name].attrs.get("units", "") for name in variable_names}


def geometry_array(da):
    da = da.squeeze(drop=True)
    if da.ndim != 2:
        raise ValueError(f"Geometry {da.name} should have two axes; found {da.dims}")
    north = next((d for d in ("yh", "lath", "lat", "latitude", "y") if d in da.dims), None)
    east = next((d for d in ("xh", "lonh", "lon", "longitude", "x") if d in da.dims), None)
    if north is not None and east is not None:
        da = da.transpose(north, east)
    elif da.shape == (nx, ny) and (nx, ny) != (ny, nx):
        da = da.transpose(*da.dims[::-1])
    if da.shape != (ny, nx):
        raise ValueError(f"Geometry {da.name} shape {da.shape} differs from sfc {(ny, nx)}")
    return np.asarray(da.values)

geometry_files = sorted(
    p for p in ARCHIVE_DIR.rglob("*.mom6.h.ocean_geometry.nc")
    if p.name.startswith(case_name + ".mom6.h.ocean_geometry")
)
geometry_path = geometry_files[0] if geometry_files else None
wet_mask = None
has_geo = False
if geometry_path is not None:
    with xr.open_dataset(geometry_path) as geometry:
        if "geolon" in geometry and "geolat" in geometry:
            lon = np.asarray(geometry_array(geometry["geolon"]), dtype=float)
            lat = np.asarray(geometry_array(geometry["geolat"]), dtype=float)
            x_grid = (lon + 180) % 360 - 180
            y_grid = lat
            x_label, y_label = "Longitude (°E)", "Latitude (°N)"
            has_geo = True
        if "wet" in geometry:
            wet_mask = geometry_array(geometry["wet"]) > 0
        elif "D" in geometry:
            wet_mask = geometry_array(geometry["D"]) > 0

if not has_geo:
    x_grid, y_grid = np.meshgrid(x_axis, y_axis)
    x_label, y_label = x_dim, y_dim

def clean_field(data):
    values = np.array(data, dtype=float, copy=True)
    values[~np.isfinite(values) | (np.abs(values) >= 1e19)] = np.nan
    if wet_mask is not None:
        values[~wet_mask] = np.nan
    return values


print("Geometry:", geometry_path if has_geo else "sfc grid coordinates")
print("Field units:", units)


Geometry: /glade/derecho/scratch/hmnguyen/archive/florida_storm_cam6_test/ocn/hist/florida_storm_cam6_test.mom6.h.ocean_geometry.nc
Field units: {'speed': 'm s-1', 'tos': 'degC', 'sos': 'psu'}


## Fix the color scales

Estimate a single robust range from up to 12 frames across the selected dates. This prevents the colorbar from changing each frame. Values beyond a limit are saturated, and gray marks missing or dry cells. `speed` is **ocean surface current speed**, not HRRR wind speed.


In [29]:
sample_positions = np.unique(np.linspace(0, len(selected) - 1, min(12, len(selected)), dtype=int))
samples = {name: [] for name in variable_names}
for position in sample_positions:
    _, path, index = selected[position]
    with xr.open_dataset(path, decode_times=True, mask_and_scale=True) as ds:
        for name in variable_names:
            values = clean_field(ds[name].isel(time=index).transpose(y_dim, x_dim).values)
            good = values[np.isfinite(values)]
            if good.size:
                samples[name].append(good)

limits = {}
for name in variable_names:
    if not samples[name]:
        raise ValueError(f"All sampled {name} values are missing; inspect the source files")
    values = np.concatenate(samples[name])
    low, high = np.percentile(values, [1, 99])
    if name == "speed":
        low = 0.0
    if not high > low:
        high = low + 1.0
    limits[name] = (float(low), float(high))
    print(f"{name:5s}: {limits[name][0]:.3g} to {limits[name][1]:.3g} {units[name]}")


speed: 0 to 1.31 m s-1
tos  : 24.1 to 30 degC
sos  : 31.8 to 36.8 psu


## Render and save

Each selected time is read as one two-dimensional slice. Frames from October 7–12 are held for 350 ms; the other frames are held for 100 ms. Change both durations in the settings cell to adjust playback. The GIF is saved in `OUTPUT_DIR`.


In [30]:
labels = {"speed": "Surface current speed", "tos": "Sea surface temperature", "sos": "Surface salinity"}
cmaps = {"speed": "magma", "tos": "turbo", "sos": "viridis"}
frames = []
durations = []

# Keep each monthly file open during rendering; xarray reads only each requested time slice.
from contextlib import ExitStack
with ExitStack() as stack:
    opened = {path: stack.enter_context(xr.open_dataset(path, decode_times=True, mask_and_scale=True))
              for path in {record[1] for record in selected}}
    for frame_number, (when, path, index) in enumerate(selected, 1):
        ds = opened[path]
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), constrained_layout=True)
        for ax, name in zip(axes, variable_names):
            values = clean_field(ds[name].isel(time=index).transpose(y_dim, x_dim).values)
            cmap = plt.get_cmap(cmaps[name]).copy()
            cmap.set_bad("#dedede")
            mesh = ax.pcolormesh(x_grid, y_grid, np.ma.masked_invalid(values),
                                 cmap=cmap, vmin=limits[name][0], vmax=limits[name][1],
                                 shading="auto", rasterized=True)
            fig.colorbar(mesh, ax=ax, shrink=0.84, label=units[name], extend="max" if name == "speed" else "both")
            ax.set(title=labels[name], xlabel=x_label, ylabel=y_label)
            ax.set_xlim(np.nanmin(x_grid), np.nanmax(x_grid))
            ax.set_ylim(np.nanmin(y_grid), np.nanmax(y_grid))
        fig.suptitle(f"{case_name}  |  {when:%Y-%m-%d %H:%M} UTC")
        image_bytes = BytesIO()
        fig.savefig(image_bytes, format="png", dpi=FIGURE_DPI)
        plt.close(fig)
        image_bytes.seek(0)
        with Image.open(image_bytes) as image:
            frames.append(image.convert("P", palette=Image.Palette.ADAPTIVE))
        durations.append(STORM_FRAME_MS if STORM_START <= when.date() <= STORM_END else OUTSIDE_FRAME_MS)
        if frame_number % 25 == 0 or frame_number == len(selected):
            print(f"Rendered {frame_number}/{len(selected)}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
gif_path = OUTPUT_DIR / (OUTPUT_NAME or f"{case_name}_surface_storm.gif")
frames[0].save(gif_path, save_all=True, append_images=frames[1:],
               duration=durations, loop=0, disposal=2, optimize=False)
print(f"Saved {gif_path} ({len(frames)} frames, {sum(durations) / 1000:.1f} s of playback)")


Rendered 25/513
Rendered 50/513
Rendered 75/513
Rendered 100/513
Rendered 125/513
Rendered 150/513
Rendered 175/513
Rendered 200/513
Rendered 225/513
Rendered 250/513
Rendered 275/513
Rendered 300/513
Rendered 325/513
Rendered 350/513
Rendered 375/513
Rendered 400/513
Rendered 425/513
Rendered 450/513
Rendered 475/513
Rendered 500/513
Rendered 513/513
Saved /glade/u/home/hmnguyen/animations/florida_storm_cam6_test_surface_storm.gif (513 frames, 80.3 s of playback)


## Preview

The GIF loops in the notebook. If plots appear flat, inspect the printed color limits and the actual value ranges in the archived `sfc` file before changing the limits.


In [ ]:
from IPython.display import Image as NotebookImage, display
display(NotebookImage(filename=str(gif_path)))
